In [ ]:
#| default_exp core.ratio

In [ ]:
#| export
from __future__ import annotations

import warnings
from numbers import Real

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

Every compression ratio in fasterai is a **fraction in [0, 1]**: `sparsity=0.5` means 50% of the
weights, `pruning_ratio=0.3` means 30% of the filters. `Sparsifier`, `SparsifyCallback`, `Pruner`,
`PruneCallback` and `SensitivityAnalyzer` all read their ratio arguments through `as_fraction`.

| You pass | fasterai reads | What happens |
|---|---|---|
| `0.4` | `0.4` | 40% |
| `0` | `0.0` | leave this one alone (per-layer dict value) |
| `1` | `1.0` | 100%, not 1% |
| `40` | `0.4` | `FutureWarning`: percents are read as `x/100` for one release |
| `150`, `-1` | — | `ValueError` |
| `'0.4'`, `True` | — | `TypeError` |

In [ ]:
#| export
_FRACTION = "a fraction in [0, 1] (0.4 = 40%)"


def as_fraction(x, name: str, layer: str | None = None, allow_zero: bool = True) -> float:
    "Read a compression ratio as a fraction in [0, 1]; a value in (1, 100] is read as a percent for one release"
    where = f"{name} for layer '{layer}'" if layer is not None else name
    if isinstance(x, bool) or not isinstance(x, Real):
        raise TypeError(f"{where} must be a number, {_FRACTION}, got {x!r}")
    x = float(x)
    if not (0 <= x <= 100):
        raise ValueError(f"{where} must be {_FRACTION}, got {x!r}")
    if x > 1:
        warnings.warn(f"{where}={x!r} looks like a percent; this argument is {_FRACTION}; "
                      f"read as {x!r}/100 = {x / 100!r} for one release — pass the fraction",
                      FutureWarning, stacklevel=2)
        x = x / 100
    if x == 0 and not allow_zero:
        raise ValueError(f"{where} must be {_FRACTION} and above 0 — a ratio of 0 removes nothing")
    return x

In [ ]:
show_doc(as_fraction)

---

## Usage

```python
from fasterai.core.ratio import as_fraction

as_fraction(0.4, 'sparsity')                       # 0.4
as_fraction(40, 'sparsity')                        # 0.4 + FutureWarning
as_fraction(0, 'pruning_ratio', layer='fc')        # 0.0 — leave 'fc' alone
as_fraction(0, 'pruning_ratio', allow_zero=False)  # ValueError: a ratio of 0 removes nothing
```

`layer=` only changes the message, so a bad value inside a per-layer dict names its layer:

```python
as_fraction(150, 'pruning_ratio', layer='layer1.0.conv1')
# ValueError: pruning_ratio for layer 'layer1.0.conv1' must be a fraction in [0, 1] (0.4 = 40%), got 150.0
```

---

## See Also

- [Sparsifier](../sparse/sparsifier.html) - `sparsity` as a fraction
- [Pruner](../prune/pruner.html) - `pruning_ratio` as a fraction
- [Sensitivity Analysis](../analyze/sensitivity.html) - per-layer targets, returned as fractions

Tests live in `nbs/tests/test_ratio.ipynb`.